# Q7: Logistic Regression vs SVM Comparison - BSDS500 Boundary Detection

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import loadmat

BASE = r"C:\Users\cqds\Downloads\bsds500archive"
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
IMAGES_TRAIN = os.path.join(BASE, "images", "train")
IMAGES_TEST = os.path.join(BASE, "images", "test")
GT_TRAIN = os.path.join(BASE, "ground_truth", "train")
GT_TEST = os.path.join(BASE, "ground_truth", "test")

### Helper functions for loading images and ground truth

In [ ]:
def list_image_files(images_dir, max_images):
    filenames = sorted(f for f in os.listdir(images_dir) if f.lower().endswith(IMAGE_EXTENSIONS))
    return filenames[:max_images]

def find_gt_path(gt_dir, image_filename):
    stem = os.path.splitext(image_filename)[0]
    candidate = os.path.join(gt_dir, stem + ".mat")
    if not os.path.exists(candidate):
        raise FileNotFoundError(
            f"No matching ground-truth .mat file found for image '{image_filename}' "
            f"(expected {candidate})"
        )
    return candidate

def load_bsds_ground_truth(mat_path):
    mat = loadmat(mat_path)
    gt_struct = mat["groundTruth"]
    n_annotators = gt_struct.shape[1]
    boundary_maps = [np.asarray(gt_struct[0, i]["Boundaries"][0, 0], dtype=float)
                      for i in range(n_annotators)]
    consensus = np.mean(boundary_maps, axis=0)
    return (consensus >= 0.5).astype(int)

def get_boundary_labels(gt_array):
    unique_vals = np.unique(gt_array)
    if len(unique_vals) <= 2:
        return (gt_array > 0).astype(int)
    diff_right = gt_array[:, :-1] != gt_array[:, 1:]
    diff_down = gt_array[:-1, :] != gt_array[1:, :]
    boundary = np.zeros_like(gt_array, dtype=bool)
    boundary[:, :-1] |= diff_right
    boundary[:-1, :] |= diff_down
    return boundary.astype(int)

def extract_pixel_features(img_array):
    gray = img_array.mean(axis=2)
    grad_y = np.abs(np.diff(gray, axis=0, prepend=gray[:1, :]))
    grad_x = np.abs(np.diff(gray, axis=1, prepend=gray[:, :1]))
    grad_mag = np.sqrt(grad_x ** 2 + grad_y ** 2)
    feats = np.stack([img_array[:, :, 0], img_array[:, :, 1], img_array[:, :, 2], gray, grad_mag], axis=-1)
    return feats.reshape(-1, 5)

def load_pixel_dataset(images_dir, gt_dir, max_images, pixels_per_image, seed=0):
    rng = np.random.RandomState(seed)
    filenames = list_image_files(images_dir, max_images)
    X_parts, y_parts = [], []
    total_pixels = 0
    for fname in filenames:
        img = np.array(Image.open(os.path.join(images_dir, fname)).convert("RGB"), dtype=float) / 255.0
        gt_path = find_gt_path(gt_dir, fname)
        gt = load_bsds_ground_truth(gt_path)
        if gt.shape != img.shape[:2]:
            raise ValueError(
                f"Ground truth shape {gt.shape} does not match image shape "
                f"{img.shape[:2]} for '{fname}'"
            )
        labels_full = get_boundary_labels(gt).reshape(-1)
        feats_full = extract_pixel_features(img)
        total_pixels += labels_full.shape[0]
        boundary_idx = np.where(labels_full == 1)[0]
        nonboundary_idx = np.where(labels_full == 0)[0]
        rng.shuffle(boundary_idx)
        rng.shuffle(nonboundary_idx)
        idx = np.concatenate([boundary_idx[:pixels_per_image], nonboundary_idx[:pixels_per_image]])
        X_parts.append(feats_full[idx])
        y_parts.append(labels_full[idx])
    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)
    return X, y, total_pixels, len(filenames)

feature_names = ["R", "G", "B", "gray", "gradient_magnitude"]

### Load training and test pixel samples

In [ ]:
X_train, y_train, total_train_pixels, n_train_images = load_pixel_dataset(
    IMAGES_TRAIN, GT_TRAIN, max_images=40, pixels_per_image=250, seed=0)
X_test, y_test, total_test_pixels, n_test_images = load_pixel_dataset(
    IMAGES_TEST, GT_TEST, max_images=15, pixels_per_image=150, seed=1)

print("Training images used:", n_train_images, " total pixels:", total_train_pixels)
print("Training sample shape:", X_train.shape, " Test sample shape:", X_test.shape)

### Standardize

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

### Logistic Regression training

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

n, d = X_train.shape
w_lr = np.zeros(d)
b_lr = 0.0
lr_rate = 0.5
n_iter = 1500

for i in range(n_iter):
    p = sigmoid(X_train.dot(w_lr) + b_lr)
    error = p - y_train
    w_lr -= lr_rate * (X_train.T.dot(error) / n)
    b_lr -= lr_rate * error.mean()

print("Logistic Regression weights:", np.round(w_lr, 3))
scores_lr = sigmoid(X_test.dot(w_lr) + b_lr)
pred_lr = (scores_lr >= 0.5).astype(int)

### Linear SVM training (hinge loss, subgradient descent)

In [ ]:
y_train_signed = np.where(y_train == 1, 1, -1)
w_svm = np.zeros(d)
b_svm = 0.0
svm_lr_rate = 0.01
svm_lambda = 0.001

for i in range(n_iter):
    margins = y_train_signed * (X_train.dot(w_svm) + b_svm)
    violates = margins < 1
    dw = 2 * svm_lambda * w_svm - (X_train[violates].T.dot(y_train_signed[violates])) / n
    db = -np.sum(y_train_signed[violates]) / n
    w_svm -= svm_lr_rate * dw
    b_svm -= svm_lr_rate * db

print("SVM weights:", np.round(w_svm, 3))
scores_svm = X_test.dot(w_svm) + b_svm
pred_svm = (scores_svm >= 0).astype(int)

### Confusion matrix counts

In [ ]:
def confusion_counts(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp, tn, fp, fn

tp_lr, tn_lr, fp_lr, fn_lr = confusion_counts(y_test, pred_lr)
tp_svm, tn_svm, fp_svm, fn_svm = confusion_counts(y_test, pred_svm)
print("Logistic Regression confusion counts:", tp_lr, tn_lr, fp_lr, fn_lr)
print("SVM confusion counts:", tp_svm, tn_svm, fp_svm, fn_svm)

### Accuracy, Precision, Recall, F1

In [ ]:
def compute_metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1

acc_lr, prec_lr, rec_lr, f1_lr = compute_metrics(tp_lr, tn_lr, fp_lr, fn_lr)
acc_svm, prec_svm, rec_svm, f1_svm = compute_metrics(tp_svm, tn_svm, fp_svm, fn_svm)

print("Metric         Logistic Regression   SVM")
print("Accuracy      ", round(acc_lr, 4), " ", round(acc_svm, 4))
print("Precision     ", round(prec_lr, 4), " ", round(prec_svm, 4))
print("Recall        ", round(rec_lr, 4), " ", round(rec_svm, 4))
print("F1-score      ", round(f1_lr, 4), " ", round(f1_svm, 4))

### ROC curve and AUC by threshold sweep

In [ ]:
def roc_curve_and_auc(y_true, scores):
    thresholds = np.sort(np.unique(scores))[::-1]
    tpr_list, fpr_list = [0.0], [0.0]
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)
    for t in thresholds[::3]:
        pred_t = (scores >= t).astype(int)
        tp, tn, fp, fn = confusion_counts(y_true, pred_t)
        tpr_list.append(tp / n_pos if n_pos > 0 else 0)
        fpr_list.append(fp / n_neg if n_neg > 0 else 0)
    tpr_list.append(1.0)
    fpr_list.append(1.0)
    fpr_arr, tpr_arr = np.array(fpr_list), np.array(tpr_list)
    order = np.argsort(fpr_arr)
    fpr_arr, tpr_arr = fpr_arr[order], tpr_arr[order]
    auc = np.trapezoid(tpr_arr, fpr_arr)
    return fpr_arr, tpr_arr, auc

fpr_lr, tpr_lr, auc_lr = roc_curve_and_auc(y_test, scores_lr)
fpr_svm, tpr_svm, auc_svm = roc_curve_and_auc(y_test, scores_svm)
print("Logistic Regression AUC:", round(auc_lr, 4))
print("SVM AUC:", round(auc_svm, 4))

### Plot ROC curves

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(fpr_lr, tpr_lr, "b-", label=f"Logistic Regression (AUC={auc_lr:.3f})")
plt.plot(fpr_svm, tpr_svm, "r-", label=f"SVM (AUC={auc_svm:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random guess (AUC=0.5)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves: Logistic Regression vs SVM\nImage boundary detection")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()